# Live demo: Агрегатор ↔ Эксплуатант

Этот notebook имитирует работу **Агрегатора**, общаясь с **запущенным** Эксплуатантом (Operator) через брокер сообщений.

- Поддерживаемые транспорты: **Mosquitto/MQTT** и **Kafka** (переключается в ноутбуке).
- Тип взаимодействия: request/response по `SystemBus`.

## Предусловия

Поднять Эксплуатанта в Docker:

- MQTT (default):

```bash
docker compose -f systems/operator/docker-compose.yml up -d --build
```

- Kafka (опционально):

```bash
cd systems/operator && make up-kafka
```

## Сценарий

1. Агрегатор отправляет заказ (`receive_order`), получает предложение (цена/срок/параметры)
2. Агрегатор подтверждает выбор Эксплуатанта (`submit_proposal`, затем `accept_order`)
3. Запуск выполнения (`start_mission`) и проверка статуса (`get_mission_status`)
4. Завершение (`complete_mission`) и финальный статус


## Диаграммы взаимодействия

### Диаграмма 1: жизненный цикл заказа

```mermaid
sequenceDiagram
  participant Customer
  participant Aggregator
  participant Operator

  Customer->>Aggregator: CreateOrder
  Aggregator->>Operator: receive_order(order)
  Operator-->>Aggregator: proposal(price,delivery_time,...)

  Aggregator->>Operator: submit_proposal(order_id)
  Operator-->>Aggregator: submitted

  Aggregator->>Operator: accept_order(order_id)
  Operator-->>Aggregator: accepted(mission_id,uas_id)

  Aggregator->>Operator: start_mission(mission_id)
  Operator-->>Aggregator: started

  loop UntilCompleted
    Aggregator->>Operator: get_mission_status(mission_id)
    Operator-->>Aggregator: status
  end

  Aggregator->>Operator: complete_mission(mission_id)
  Operator-->>Aggregator: completed
```

### Диаграмма 2: envelope request/response

```mermaid
sequenceDiagram
  participant Aggregator
  participant Bus
  participant Operator

  Aggregator->>Bus: request(topic, {action,payload,sender})
  Bus->>Operator: publish(topic, message+{correlation_id,reply_to})
  Operator->>Bus: publish(reply_to, response)
  Bus-->>Aggregator: response({correlation_id,payload,success,error?})
```


### PNG (PlantUML) диаграммы

![](assets/plantuml/operator_components_updated.png)

![](assets/plantuml/uas_purchase_sequence.png)

![](assets/plantuml/fleet_manager_architecture.png)

![](assets/plantuml/uas_reservation_sequence.png)


In [32]:
# Настройки (выберите брокер: 'mqtt' или 'kafka')
import os
import sys
from pathlib import Path

# В VSCode notebook обычно cwd=notebooks/.
# Добавляем корень репозитория в sys.path, чтобы работали импорты `broker`, `systems`, `sdk`.
repo_root = Path.cwd().resolve()
while repo_root != repo_root.parent and not (repo_root / "broker").exists():
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root))

broker = os.getenv("AGG_BROKER", "mqtt")  # "mqtt" | "kafka"

SYSTEM_ID = os.getenv("SYSTEM_ID", "operator-001")
API_VERSION = os.getenv("API_VERSION", "v1")

# MQTT
MQTT_BROKER = os.getenv("MQTT_BROKER", "localhost")
MQTT_PORT = int(os.getenv("MQTT_PORT", "1883"))

# Kafka (для ноутбука используем внешний listener)
KAFKA_BOOTSTRAP_SERVERS = os.getenv("KAFKA_BOOTSTRAP_SERVERS", "localhost:19092")

OPERATOR_TOPIC = f"{SYSTEM_ID}.{API_VERSION}.operator"

print("repo_root=", repo_root)
print("broker=", broker)
print("OPERATOR_TOPIC=", OPERATOR_TOPIC)
print("MQTT=", f"{MQTT_BROKER}:{MQTT_PORT}")
print("KAFKA=", KAFKA_BOOTSTRAP_SERVERS)


repo_root= /home/user/projects/sbd-drones-economics/sbd-drones-economics-ai
broker= mqtt
OPERATOR_TOPIC= operator-001.v1.operator
MQTT= localhost:1883
KAFKA= localhost:19092


In [33]:
import time
from dataclasses import dataclass


@dataclass
class BusConfig:
    broker: str
    bus: object


def create_bus(selected: str) -> BusConfig:
    selected = selected.lower().strip()

    if selected == "mqtt":
        from broker.mqtt.mqtt_system_bus import MQTTSystemBus

        b = MQTTSystemBus(broker=MQTT_BROKER, port=MQTT_PORT, client_id=f"aggregator.{SYSTEM_ID}")
        b.start()
        return BusConfig(broker="mqtt", bus=b)

    if selected == "kafka":
        from broker.kafka.kafka_system_bus import KafkaSystemBus

        b = KafkaSystemBus(
            bootstrap_servers=KAFKA_BOOTSTRAP_SERVERS,
            client_id=f"aggregator.{SYSTEM_ID}",
            group_id=f"aggregator-{SYSTEM_ID}",
        )
        b.start()
        return BusConfig(broker="kafka", bus=b)

    raise ValueError(f"Unknown broker: {selected}")


bus_cfg = create_bus(broker)
bus = bus_cfg.bus
print("Bus started:", bus_cfg.broker)


MQTTSystemBus connected to localhost:1883
MQTTSystemBus started. Reply topic: replies/aggregator.operator-001_5300cffb
Bus started: mqtt


In [34]:
# Поднять нужный docker-compose стек (через Makefile)
# Важно: требуется установленный Docker.

import subprocess

if broker == "mqtt":
    cmd = ["make", "-C", str(repo_root / "systems" / "operator"), "up-mqtt"]
elif broker == "kafka":
    cmd = ["make", "-C", str(repo_root / "systems" / "operator"), "up-kafka"]
else:
    raise ValueError(f"Unknown broker: {broker}")

print("Running:", " ".join(cmd))
subprocess.run(cmd, check=True)


Running: make -C /home/user/projects/sbd-drones-economics/sbd-drones-economics-ai/systems/operator up-mqtt
make: Entering directory '/home/user/projects/sbd-drones-economics/sbd-drones-economics-ai/systems/operator'
docker build -t operator-base:latest -f ../../systems/operator/docker/Dockerfile.base ../..; \
docker compose -f docker-compose.yml up -d --build --remove-orphans


/etc/bash.bashrc: line 7: PS1: unbound variable
#0 building with "default" instance using docker driver

#1 [internal] load build definition from Dockerfile.base
#1 DONE 0.0s

#1 [internal] load build definition from Dockerfile.base
#1 transferring dockerfile: 825B done
#1 DONE 0.0s

#2 [internal] load metadata for docker.io/library/python:3.12.3-slim
#2 DONE 0.8s

#3 [internal] load .dockerignore
#3 transferring context: 2B done
#3 DONE 0.0s

#4 [1/7] FROM docker.io/library/python:3.12.3-slim@sha256:afc139a0a640942491ec481ad8dda10f2c5b753f5c969393b12480155fe15a63
#4 DONE 0.0s

#5 [internal] load build context
#5 transferring context: 499B done
#5 DONE 0.0s

#6 [2/7] WORKDIR /app
#6 CACHED

#7 [5/7] RUN pip install --no-cache-dir -r requirements.runtime.txt
#7 CACHED

#8 [6/7] RUN if [ "0" = "1" ]; then pip install --no-cache-dir -r requirements.kafka.txt; fi
#8 CACHED

#9 [3/7] COPY systems/operator/requirements.runtime.txt ./requirements.runtime.txt
#9 CACHED

#10 [4/7] COPY systems/

#1 [internal] load local bake definitions
#1 reading from stdin 3.20kB done
#1 DONE 0.0s

#2 [fleet-manager internal] load build definition from Dockerfile
#2 transferring dockerfile: 799B done
#2 DONE 0.0s

#3 [business-logic internal] load metadata for docker.io/library/operator-base:latest
#3 DONE 0.0s

#4 [fleet-manager internal] load .dockerignore
#4 transferring context: 2B done
#4 DONE 0.0s

#5 [security-monitor 1/6] FROM docker.io/library/operator-base:latest
#5 DONE 0.0s

#6 [security-monitor internal] load build context
#6 transferring context: 960.11kB 0.0s done
#6 transferring context: 960.11kB 0.0s done
#6 DONE 0.0s

#7 [fleet-manager 2/6] WORKDIR /app
#7 CACHED

#8 [fleet-manager 3/6] COPY systems/operator/src/ ./systems/operator/src/
#8 DONE 0.0s

#9 [operator-system 4/6] COPY systems/operator/__init__.py ./systems/operator/__init__.py
#9 DONE 0.0s

#10 [fleet-manager 5/6] COPY sdk/ ./sdk/
#10 DONE 0.0s

#11 [operator-system 6/6] COPY broker/ ./broker/
#11 DONE 0.0s

#12

 Image operator-security-monitor Built 
 Image operator-business-logic Built 
 Image operator-fleet-manager Built 
 Image operator-mission-planner Built 
 Image operator-operator-system Built 
 Container operator-mosquitto Running 
 Container operator-security-monitor Recreate 
 Container operator-security-monitor Recreated 
 Container operator-fleet-manager Recreate 
 Container operator-mission-planner Recreate 
 Container operator-business-logic Recreate 
 Container operator-business-logic Recreated 
 Container operator-fleet-manager Recreated 
 Container operator-mission-planner Recreated 
 Container operator-system Recreate 
 Container operator-system Recreated 
 Container operator-security-monitor Starting 
 Container operator-security-monitor Started 
 Container operator-fleet-manager Starting 
 Container operator-mission-planner Starting 
 Container operator-business-logic Starting 


make: Leaving directory '/home/user/projects/sbd-drones-economics/sbd-drones-economics-ai/systems/operator'


 Container operator-fleet-manager Started 
 Container operator-business-logic Started 
 Container operator-mission-planner Started 
 Container operator-system Starting 
 Container operator-system Started 


CompletedProcess(args=['make', '-C', '/home/user/projects/sbd-drones-economics/sbd-drones-economics-ai/systems/operator', 'up-mqtt'], returncode=0)

In [35]:
# Ожидание готовности контейнеров (healthcheck/running)

import time


def _docker_inspect(fmt: str, name: str) -> str:
    # Returns empty string on error.
    p = subprocess.run(
        ["docker", "inspect", "-f", fmt, name],
        capture_output=True,
        text=True,
    )
    if p.returncode != 0:
        return ""
    return (p.stdout or "").strip()


def wait_container(name: str, timeout_s: int = 180) -> None:
    deadline = time.time() + timeout_s
    last = None
    while time.time() < deadline:
        health = _docker_inspect("{{.State.Health.Status}}", name)
        state = _docker_inspect("{{.State.Status}}", name)
        status = health or state
        if status and status in {"healthy", "running"}:
            print(f"OK: {name} status={status}")
            return
        last = (health, state)
        time.sleep(2)
    raise TimeoutError(f"Timeout waiting {name}: health/state={last}")


mqtt_containers = [
    "operator-mosquitto",
    "operator-security-monitor",
    "operator-fleet-manager",
    "operator-mission-planner",
    "operator-business-logic",
    "operator-system",
]

kafka_containers = [
    "operator-kafka",
    "operator-security-monitor",
    "operator-fleet-manager",
    "operator-mission-planner",
    "operator-business-logic",
    "operator-system",
]

containers = mqtt_containers if broker == "mqtt" else kafka_containers

print("Waiting containers:", containers)
for c in containers:
    wait_container(c, timeout_s=240)


Waiting containers: ['operator-mosquitto', 'operator-security-monitor', 'operator-fleet-manager', 'operator-mission-planner', 'operator-business-logic', 'operator-system']
OK: operator-mosquitto status=healthy
OK: operator-security-monitor status=healthy
OK: operator-fleet-manager status=healthy
OK: operator-mission-planner status=healthy
OK: operator-business-logic status=healthy
OK: operator-system status=healthy


In [36]:
# Проверка доступности брокера (readiness)

import time


def wait_broker_ready(timeout_s: float = 30.0) -> None:
    deadline = time.time() + timeout_s
    last_exc: Exception | None = None

    while time.time() < deadline:
        try:
            if broker == "mqtt":
                from broker.mqtt.mqtt_system_bus import MQTTSystemBus

                b = MQTTSystemBus(broker=MQTT_BROKER, port=MQTT_PORT, client_id=f"nb-ready.{SYSTEM_ID}")
                b.start()
                b.stop()
                print("OK: MQTT broker ready")
                return

            if broker == "kafka":
                from broker.kafka.kafka_system_bus import KafkaSystemBus

                b = KafkaSystemBus(
                    bootstrap_servers=KAFKA_BOOTSTRAP_SERVERS,
                    client_id=f"nb-ready.{SYSTEM_ID}",
                    group_id=f"nb-ready-{SYSTEM_ID}",
                )
                b.start()
                b.stop()
                print("OK: Kafka broker ready")
                return

            raise ValueError(f"Unknown broker: {broker}")

        except Exception as e:  # noqa: BLE001
            last_exc = e
            time.sleep(0.5)

    raise TimeoutError(f"Broker not ready after {timeout_s}s: {last_exc}")


wait_broker_ready(45.0)


MQTTSystemBus connected to localhost:1883
MQTTSystemBus started. Reply topic: replies/nb-ready.operator-001_1f0fd1b6
MQTTSystemBus stopped
OK: MQTT broker ready


In [37]:
def request(action: str, payload: dict, timeout_s: float = 30.0, *, retries: int = 8) -> dict:
    """Aggregator -> Operator request/response with retry.

    Ретрай только по таймаутам брокера/сервиса; бизнес-ошибки возвращаем вызывающему коду.
    """

    last_exc: Exception | None = None

    for attempt in range(1, retries + 1):
        try:
            resp = bus.request(
                OPERATOR_TOPIC,
                {
                    "action": action,
                    "sender": "aggregator-demo",
                    "payload": payload,
                },
                timeout=timeout_s,
            )
            if resp is None:
                raise TimeoutError(f"No response for action={action}")

            payload_resp = resp.get("payload", {})
            # Бизнес-ошибки не считаем поводом для retry: просто возвращаем payload наверх.
            return payload_resp

        except TimeoutError as e:
            last_exc = e
            if attempt == retries:
                break
            sleep_s = min(8.0, 0.5 * (2 ** (attempt - 1)))
            print(
                f"WARN: timeout for action={action} attempt={attempt}/{retries}: {e}. "
                f"Retry in {sleep_s:.1f}s"
            )
            time.sleep(sleep_s)

        except Exception as e:  # noqa: BLE001
            last_exc = e
            break

    print("Diagnostics:")
    print("  broker=", broker)
    print("  OPERATOR_TOPIC=", OPERATOR_TOPIC)
    if broker == "mqtt":
        print("  MQTT=", f"{MQTT_BROKER}:{MQTT_PORT}")
    else:
        print("  KAFKA=", KAFKA_BOOTSTRAP_SERVERS)
    return {"error": str(last_exc) if last_exc else "request failed"}


def pretty(d: dict) -> None:
    import json

    print(json.dumps(d, ensure_ascii=False, indent=2))


In [38]:
# 1) Создаём заказ и получаем предложение
# Если парк пустой — автоматически покупаем 1 БАС и повторяем запрос.


def ensure_fleet_has_uas() -> None:
    lst = request("GET_UAS_LIST", {}, timeout_s=15.0)
    total = lst.get("total") or lst.get("total_count") or 0
    if total:
        return

    catalogs = request("GET_DEVELOPER_CATALOGS", {}, timeout_s=20.0)
    cats = catalogs.get("catalogs") or {}
    if not cats:
        raise RuntimeError(f"No developer catalogs available: {catalogs}")

    # pick first model with availability
    pick_dev = None
    pick_model = None
    for dev_id, cat in cats.items():
        for m in cat.get("models", []):
            if int(m.get("available_quantity", 0) or 0) > 0:
                pick_dev = dev_id
                pick_model = m.get("model_id")
                break
        if pick_dev:
            break

    if not pick_dev or not pick_model:
        raise RuntimeError(f"No available models in catalogs: {list(cats.keys())}")

    pur = request(
        "PURCHASE_UAS",
        {"developer_id": pick_dev, "model_id": pick_model, "quantity": 1},
        timeout_s=30.0,
    )
    if pur.get("success") is not True:
        raise RuntimeError(f"PURCHASE_UAS failed: {pur}")


order_id = f"ORDER-LIVE-{int(time.time())}"
order = {
    "id": order_id,
    "pickup": {"lat": 55.76, "lon": 37.62},
    "dropoff": {"lat": 55.75, "lon": 37.61},
    "payload_weight": 1.0,
    "distance_km": 2.0,
    "payload_value": 20000,
}

receive_res = request("receive_order", {"order": order}, timeout_s=30.0)
if receive_res.get("proposal", {}).get("error") == "No suitable UAS available":
    print("Fleet empty/no suitable UAS. Purchasing one and retrying...")
    ensure_fleet_has_uas()
    receive_res = request("receive_order", {"order": order}, timeout_s=30.0)

print("receive_order response")
pretty(receive_res)

proposal = receive_res.get("proposal", {})
print("proposal")
pretty(proposal)


Fleet empty/no suitable UAS. Purchasing one and retrying...


RuntimeError: No developer catalogs available: {'error': 'Unknown action: GET_DEVELOPER_CATALOGS', 'trace_id': '1a6d6f25-5830-4095-b57f-75925a709a4d', 'span_id': 'de3d951b-503b-4a5f-ac3d-f5f8d1d0d700', 'parent_span_id': '156a0af2-a537-438e-9692-a30aac917512'}

In [ ]:
# 2) Подтверждаем, что предложение отправлено (submit_proposal)
if "error" in proposal:
    raise RuntimeError(f"Proposal has error, cannot continue: {proposal}")

submit_res = request("submit_proposal", {"order_id": order_id}, timeout_s=15.0)
print("submit_proposal response")
pretty(submit_res)

if "error" in submit_res:
    raise RuntimeError(f"submit_proposal failed: {submit_res}")

# 3) Выбираем Эксплуатанта исполнителем (accept_order)
accept_res = request("accept_order", {"order_id": order_id}, timeout_s=30.0)
print("accept_order response")
pretty(accept_res)

if "error" in accept_res:
    raise RuntimeError(f"accept_order failed: {accept_res}")

mission_id = accept_res.get("mission_id")
if not mission_id:
    raise RuntimeError(f"accept_order did not return mission_id: {accept_res}")


In [ ]:
# 4) Запускаем выполнение и опрашиваем статус
start_res = request("start_mission", {"mission_id": mission_id}, timeout_s=30.0)
print("start_mission response")
pretty(start_res)

status = None
for _ in range(10):
    status = request("get_mission_status", {"mission_id": mission_id}, timeout_s=15.0)
    print("get_mission_status")
    pretty(status)
    if status.get("status") in {"completed", "failed", "aborted"}:
        break
    time.sleep(1.0)

# 5) Завершаем миссию (если ещё не завершена)
if status and status.get("status") != "completed":
    complete_res = request("complete_mission", {"mission_id": mission_id}, timeout_s=30.0)
    print("complete_mission response")
    pretty(complete_res)

final_status = request("get_mission_status", {"mission_id": mission_id}, timeout_s=15.0)
print("final get_mission_status")
pretty(final_status)


In [ ]:
# Cleanup (по желанию)
# bus.stop()
# print("Bus stopped")
